# Atenciones de urgencia 2024: una vista por columna

Fuente: DEIS-MINSAL (`AtencionesUrgencia2024.zip`, 8,97 millones de filas, 21 columnas).

**Por qué no sirve contar filas.** Cada establecimiento reporta todas las causas todos los días, así que el número de filas por región, causa, semana o fecha es casi igual en todas. Las diferencias reales están en `Total` (atenciones). Por eso aquí cada columna se grafica **ponderada por atenciones**.

**Por qué se filtra.** Las causas son jerárquicas (hay totales, subtotales y detalle en el mismo archivo). Para no contar dos veces:
- Para describir *dónde y cuándo* se atiende, se usa solo la fila `IdCausa == 1` (total de atenciones de urgencia del establecimiento ese día).
- Para describir *por qué* se atiende, se usan solo las causas de detalle.

In [ ]:
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

ZIP, CSV = "AtencionesUrgencia2024.zip", "AtencionesUrgencia2024.csv"
EDADES = ["Menores_1", "De_1_a_4", "De_5_a_14", "De_15_a_64", "De_65_y_mas"]
NOMBRE_EDAD = dict(zip(EDADES, ["Menores de 1", "1 a 4", "5 a 14", "15 a 64", "65 y más"]))
TEXTO = ["IdEstablecimiento", "NEstablecimiento", "GlosaCausa", "GLOSATIPOESTABLECIMIENTO",
         "GLOSATIPOATENCION", "GlosaTipoCampana", "CodigoRegion", "NombreRegion",
         "CodigoDependencia", "NombreDependencia", "CodigoComuna", "NombreComuna"]

# Causas de detalle que no se solapan entre sí (21 = "Total demás causas", sin desglose en el archivo)
DETALLE = [3, 4, 5, 6, 10, 11, 13, 14, 15, 16, 17, 19, 20, 21, 29, 30, 31, 35, 37, 38, 39, 40, 41]
# Grupos que suman el total de la Sección 1
GRUPOS = {"Respiratorio": [2], "Demás causas": [21], "Traumatismos": [18], "Diarrea aguda": [29],
          "Circulatorio": [12], "Salud mental": [36], "COVID-19": [30, 31]}

AZUL = "#2a78d6"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
miles = mtick.FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", "."))
def corto(v):
    return f"{v/1e6:.1f} M".replace(".", ",") if v >= 1e6 else f"{v/1e3:.0f} mil" if v >= 1e3 else f"{v:.0f}"

## 1. Carga

In [ ]:
dtypes = {c: "category" for c in TEXTO}
dtypes.update({c: "int32" for c in ["IdCausa", "Total", "semana"] + EDADES})

with zipfile.ZipFile(ZIP) as z, z.open(CSV) as f:
    df = pd.read_csv(f, sep=";", encoding="latin1", dtype=dtypes,
                     usecols=["IdCausa", "Total", "semana", "fecha"] + EDADES + TEXTO)
df["fecha"] = pd.to_datetime(df["fecha"], format="%d/%m/%Y")

tot = df[df["IdCausa"] == 1].copy()          # una fila por establecimiento y día
det = df[df["IdCausa"].isin(DETALLE)]        # causas de detalle
print(f"{len(df):,} filas | {len(tot):,} filas de total | {tot['Total'].sum():,} atenciones".replace(",", "."))

## 2. Columnas numéricas

- `Total`: atenciones por establecimiento y día (escala log). Se separa por tipo de establecimiento porque ahí está la diferencia: un hospital grande y un SUR rural no se parecen.
- `semana`: atenciones totales por semana del año.
- Tramos de edad: qué porcentaje de las atenciones del día corresponde a cada tramo.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(12, 14))

# Total por establecimiento-día, por tipo de establecimiento
ax = axes[0, 0]
bins = np.logspace(0, np.log10(tot["Total"].max()), 50)
for tipo, c in zip(["Hospital", "SAPU", "SAR", "SUR"], ["#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7"]):
    v = tot.loc[(tot["GLOSATIPOESTABLECIMIENTO"] == tipo) & (tot["Total"] > 0), "Total"]
    ax.hist(v, bins=bins, histtype="step", linewidth=1.8, color=c, label=f"{tipo} (mediana {v.median():.0f})")
ax.set_xscale("log"); ax.set_ylabel("establecimiento-días")
ax.set_title("Total: atenciones por establecimiento y día", fontsize=10); ax.legend(fontsize=8, frameon=False)
ax.yaxis.set_major_formatter(miles)

# semana
ax = axes[0, 1]
por_sem = tot.groupby("semana")["Total"].sum()
ax.bar(por_sem.index, por_sem.values, color=AZUL)
ax.set_title("semana: atenciones totales", fontsize=10); ax.set_xlabel("semana del año")
ax.yaxis.set_major_formatter(miles)

# edades: % del total del día
tt = tot[tot["Total"] > 0]
for ax, col in zip(axes.flat[2:], EDADES):
    pct = tt[col] / tt["Total"] * 100
    ax.hist(pct, bins=np.linspace(0, 100, 41), color=AZUL)
    ax.set_yscale("log")
    ax.set_title(f"{col}: % de las atenciones del día  (media {pct.mean():.0f} %)", fontsize=10)
    ax.set_xlabel("% del total diario"); ax.set_ylabel("establecimiento-días")
for ax in axes.flat[7:]:
    ax.axis("off")
fig.tight_layout(); plt.show()

## 3. Columnas de texto y códigos: atenciones por categoría

Top 15 por atenciones totales del año.

In [ ]:
def top_barras(ax, serie, titulo, n=15, etiqueta=lambda x: str(x)[:42]):
    top = serie.sort_values(ascending=False).head(n)[::-1]
    ax.barh([etiqueta(i) for i in top.index], top.values, color=AZUL)
    ax.set_title(f"{titulo}  |  {len(serie):,} categorías".replace(",", "."), fontsize=10)
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: corto(v)))
    ax.tick_params(axis="y", labelsize=8)

COLS_LUGAR = ["IdEstablecimiento", "NEstablecimiento", "GLOSATIPOESTABLECIMIENTO", "GLOSATIPOATENCION",
              "GlosaTipoCampana", "CodigoRegion", "NombreRegion", "CodigoDependencia",
              "NombreDependencia", "CodigoComuna", "NombreComuna"]
fig, axes = plt.subplots(6, 2, figsize=(14, 24))
for ax, col in zip(axes.flat, COLS_LUGAR):
    top_barras(ax, tot.groupby(col, observed=True)["Total"].sum(), col)
for ax in axes.flat[len(COLS_LUGAR):]:
    ax.axis("off")
fig.tight_layout(); plt.show()

In [ ]:
# Causas: IdCausa y GlosaCausa son el mismo dato (código y nombre). Solo causas de detalle.
por_causa = det.groupby(["IdCausa", "GlosaCausa"], observed=True)["Total"].sum()
por_causa = por_causa[por_causa > 0].reset_index()
por_causa["GlosaCausa"] = por_causa["GlosaCausa"].astype(str).str.strip(" -\xa0")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
top_barras(axes[0], por_causa.set_index("GlosaCausa")["Total"], "GlosaCausa (causas de detalle)", n=15)
top_barras(axes[1], por_causa.set_index("IdCausa")["Total"], "IdCausa (causas de detalle)", n=15, etiqueta=lambda x: f"causa {x}")
fig.tight_layout(); plt.show()

## 4. Fecha

Atenciones por día (total nacional) con promedio móvil de 7 días.

In [ ]:
dia = tot.groupby("fecha")["Total"].sum()
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.bar(dia.index, dia.values, width=1, color="#b9c9dc", label="día")
ax.plot(dia.index, dia.rolling(7, center=True).mean(), color=AZUL, linewidth=2, label="promedio móvil 7 días")
ax.set_title(f"fecha: atenciones por día  |  máx. {dia.max():,} el {dia.idxmax():%d-%m}, mín. {dia.min():,} el {dia.idxmin():%d-%m}".replace(",", "."), fontsize=10)
ax.yaxis.set_major_formatter(miles); ax.legend(frameon=False)
fig.tight_layout(); plt.show()

## 5. Dónde sí se ven las diferencias

Comparar grupos de causa: la estacionalidad y la edad cambian mucho de un grupo a otro.

In [ ]:
nombre = {i: n for n, ids in GRUPOS.items() for i in ids}
g = df[df["IdCausa"].isin(nombre)].assign(grupo=lambda d: d["IdCausa"].map(nombre))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
semanal = g.pivot_table(index="semana", columns="grupo", values="Total", aggfunc="sum", observed=True)
for col in ["Respiratorio", "Traumatismos", "Diarrea aguda", "Salud mental", "Circulatorio"]:
    axes[0].plot(semanal.index, semanal[col] / semanal[col].mean(), linewidth=2, label=col)
axes[0].axhline(1, color="gray", linewidth=0.8)
axes[0].set_title("Estacionalidad: atenciones semanales / promedio del grupo", fontsize=10)
axes[0].set_xlabel("semana del año"); axes[0].legend(frameon=False, fontsize=8)

orden = ["Respiratorio", "Diarrea aguda", "Traumatismos", "Salud mental", "Circulatorio"]
edad = g.groupby("grupo", observed=True)[EDADES].sum()
edad = edad.div(edad.sum(axis=1), axis=0).loc[orden][::-1] * 100
izq = np.zeros(len(edad))
for col, c in zip(EDADES, ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]):
    axes[1].barh(edad.index, edad[col], left=izq, color=c, label=NOMBRE_EDAD[col], edgecolor="white")
    izq += edad[col].values
axes[1].set_title("Edad de los pacientes por grupo de causa (%)", fontsize=10)
axes[1].legend(frameon=False, fontsize=8, ncol=5, loc="lower center", bbox_to_anchor=(0.5, -0.22))
fig.tight_layout(); plt.show()